# Comparaison d'algos en Classification Supervisée

Nous allons comparer les différentes méthodes de classification supervisée que nous avons vues jusqu'à présent, la régression logistique avec toutes les variables, puis avec les variables sélectionnées par le critère BIC avec un algo backward et par AIC puis les méthodes de vraisemblance pénalisée.
Le nombre de bloc de la validation croisée vaut $k=10$ mais peut être modifié par l'utilisateur. Les données s'appellent *don* et la variable d'intérêt $Y$

In [2]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
import sklearn.metrics as sklm
from patsy import dmatrix

import logistic_step_sk as lss

In [3]:
# dfbase.csv est généré par preparation.ipynb :
# les variables qualitatives y sont déjà encodées en dummies,
# et la variable cible est renommée Y
# Alternative : charger directement depuis l'URL (sans passer par preparation.ipynb) :
# don = pd.read_csv("https://regression-avec-python.github.io/donnees/SAh.csv", header=0, sep=",")
# don.rename(columns={"chd":"Y"}, inplace=True)
don = pd.read_csv("dfbase.csv", header=0, sep=",")
don.head(3)

,sbp,tobacco,ldl,adiposity,typea,obesity,alcohol,age,famhist_Present,Y
0,160,12.00,5.73,23.11,49,25.30,97.20,52,1.0,1
1,144,0.01,4.41,28.61,55,28.87,2.06,63,0.0,1
2,118,0.08,3.48,32.28,52,29.14,3.81,46,1.0,0


In [4]:
# On affiche les noms des variables explicatives pour vérification
nomsvar = list(don.columns.difference(["Y"]))
print("Variables explicatives :", nomsvar)

# dfbase.csv contient déjà les dummies => conversion directe en numpy pour sklearn
X = don.drop(columns=["Y"]).to_numpy()
Y = don["Y"].to_numpy()

Variables explicatives : ['adiposity', 'age', 'alcohol', 'famhist_Present', 'ldl', 'obesity', 'sbp', 'tobacco', 'typea']


## Préparation Validation croisée

In [5]:
nb = 10
# StratifiedKFold découpe en 10 folds en préservant la proportion de Y=1/Y=0 dans chaque fold
skf = StratifiedKFold(n_splits=nb, shuffle=True, random_state=123)

# df PROB : création du tableau de résultats - stocke les probabilité de chaque modèle
PROB = pd.DataFrame({"Y": Y, "log": 0.0, "BIC": 0.0, "AIC": 0.0,
                     "ridge": 0.0, "lasso": 0.0, "elast": 0.0,
                     "arbre": 0.0, "foret": 0.0})

## Choix des grilles de régularisation

Calcule une grille de valeurs du paramètre de régularisation C (= inverse de  λ) adaptée à l'échelle des données.
C'est la valeur au-delà de laquelle le Lasso met tous les coefficients à zéro. La grille explore 100 valeurs de 
λ0 jusqu'à λ0×10−4. Ridge et ElasticNet ont des grilles décalées (×100 et ×2) car leurs échelles diffèrent.

In [6]:
def grille(X, y, type="lasso", ng=100):
    """Calcule une grille de valeurs C (inverse du paramètre de régularisation)
    adaptée à l'échelle des données, en partant de la plus grande valeur utile l0
    (en dessous de laquelle tous les coefficients sont nuls pour le lasso)."""
    scalerX = StandardScaler().fit(X)
    Xcr = scalerX.transform(X)
    l0 = np.abs(Xcr.transpose().dot((y - y.mean()))).max() / X.shape[0]
    llc = np.linspace(0, -4, ng)
    if type == "lasso":
        Cs = 1 / 0.9 / X.shape[0] / (l0 * 10 ** (llc))
    elif type == "ridge":
        # Ridge nécessite une grille 100x plus grande que lasso
        Cs = 1 / 0.9 / X.shape[0] / ((l0 * 10 ** (llc)) * 100)
    elif type == "enet":
        Cs = 1 / 0.9 / X.shape[0] / ((l0 * 10 ** (llc)) * 2)
    return Cs

## On compare les méthodes en utilisant skf

Pour chaque fold (app_index, val_index) de skf.split(X, Y) :

| Méthode | Description |
|---|---|
| log	| Régression logistique complète, sans pénalité (penalty=None) |
| BIC	| Sélection forward de variables minimisant le BIC (via logistic_step_sk) |
| AIC	| Idem avec le critère AIC |
| lasso	| Logistique pénalisée L1 — sélectionne des variables (coefficients → 0) |
| elast	| Elastic Net (compromis L1/L2, l1_ratio=0.5) |
| ridge	| Logistique pénalisée L2 — réduit les coefficients sans les annuler |
| arbre	| Arbre de décision (min_samples_leaf=5 pour éviter le surapprentissage) |
| foret	| Forêt aléatoire (ensemble d'arbres) |

In [7]:
for app_index, val_index in skf.split(X, Y):
    Xapp  = X[app_index, :]   # toutes les colonnes, lignes d'apprentissage
    Xtest = X[val_index, :]   # toutes les colonnes, lignes de test
    Yapp  = Y[app_index]      # labels d'apprentissage uniquement

    ### logistique complète (toutes variables, sans pénalité)
    log = LogisticRegression(penalty=None, solver="newton-cholesky").fit(Xapp, Yapp)
    PROB.loc[val_index, "log"] = log.predict_proba(Xtest)[:, 1]     # on stocke la proba d'appartenir à la classe Y=1

    ### sélection forward par BIC
    choixbic = lss.LogisticRegressionSelectionFeatureIC(
        start=[], direction="forward", crit="bic", multi_class="auto"
    ).fit(Xapp, Yapp)
    PROB.loc[val_index, "BIC"] = choixbic.predict_proba(Xtest)[:, 1]

    ### sélection forward par AIC
    choixaic = lss.LogisticRegressionSelectionFeatureIC(
        start=[], direction="forward", crit="aic", multi_class="auto"
    ).fit(Xapp, Yapp)
    PROB.loc[val_index, "AIC"] = choixaic.predict_proba(Xtest)[:, 1]

    ### lasso logistique (pénalité L1 — sélection de variables)
    cr = StandardScaler()
    Cs_lasso = grille(Xapp, Yapp, "lasso")      
    lassocv = LogisticRegressionCV(
        cv=10, penalty="l1", n_jobs=10, Cs=Cs_lasso, solver="saga", max_iter=2000
    )
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp, Yapp)
    PROB.loc[val_index, "lasso"] = pipe_lassocv.predict_proba(Xtest)[:, 1]

    ### elastic net (compromis L1/L2, l1_ratio=0.5)
    cr = StandardScaler()
    Cs_enet = grille(Xapp, Yapp, "enet")
    enetcv = LogisticRegressionCV(
        cv=10, penalty="elasticnet", n_jobs=10, l1_ratios=[0.5],
        Cs=Cs_enet, solver="saga", max_iter=2000
    )
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp, Yapp)
    PROB.loc[val_index, "elast"] = pipe_enetcv.predict_proba(Xtest)[:, 1]

    ### ridge logistique (pénalité L2 — shrinkage sans sélection)
    cr = StandardScaler()
    Cs_ridge = grille(Xapp, Yapp, "ridge")
    ridgecv = LogisticRegressionCV(
        cv=10, penalty="l2", Cs=Cs_ridge, max_iter=1000
    )
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp, Yapp)
    PROB.loc[val_index, "ridge"] = pipe_ridgecv.predict_proba(Xtest)[:, 1]

    ### arbre de décision (min 5 observations par feuille pour éviter le surapprentissage)
    arbre = DecisionTreeClassifier(min_samples_leaf=5).fit(Xapp, Yapp)
    PROB.loc[val_index, "arbre"] = arbre.predict_proba(Xtest)[:, 1]

    ### forêt aléatoire
    foret = RandomForestClassifier().fit(Xapp, Yapp)
    PROB.loc[val_index, "foret"] = foret.predict_proba(Xtest)[:, 1]

/home/user/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/user/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/user/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/user/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was depr

## Résultats

In [8]:
# Aperçu des probabilités prédites pour les 4 premières observations
round(PROB.iloc[0:4, :], 3)

,Y,log,BIC,AIC,ridge,lasso,elast,arbre,foret
0,1,0.742,0.689,0.689,0.677,0.598,0.605,1.000,0.76
1,1,0.292,0.362,0.333,0.317,0.373,0.358,0.000,0.25
2,0,0.251,0.313,0.230,0.286,0.314,0.297,0.667,0.13
3,1,0.719,0.720,0.681,0.679,0.696,0.682,1.000,0.79


In [9]:
# Sauvegarde des résultats selon le type de features utilisé dans preparation.ipynb
PROB.to_csv("PROB.csv", index=False)        # features de base (dfbase)

In [13]:
PROB.to_csv("PROBpoly.csv", index=False)    # features polynomiales (dfpoly)

In [14]:
PROB.to_csv("PROBinter.csv", index=False)   # features avec interactions (dfinter)

In [ ]:
PROB.to_csv("PROBfull.csv", index=False)   # features avec interactions (dffull)